Rewrite of script by same name in notebook format, for testing gurobipy code etc. 

In [1]:
import numpy as np
import time
import os
from pathlib import Path
import corono as coro
from astropy.io import fits

# Parameters

Telescope name

In [2]:
pupil_name   = 'sbr'
problem_name = 'MaxTau'
solver       = 'gurobipy'

In [3]:
slvLogToConsole = 0
slvCrossover    = 0
slvMethod       = 2
allLogToConsole = 0

In [4]:
MinIsland   = False
Binarity    = False
FirstDerGlobalLim = 100.
BinarityReg       = 0.1

nPup = corono0.params['nPup']

In [5]:
nPup = 50
nFPM = 50
Fmax2d = 22.5
nImg2d = 45

mask radius in lam0/D units

In [6]:
rMask = 4.0

dark zone bounds (inner and outer edges) in lam0/D unit

In [7]:
rho0 =  5.0
rho1 = 10.0

contrast in the dark region

In [8]:
cDarkHole = 7.0

tau (integrated Pupil transmission)

In [9]:
tau   = 0.4

CtrBtwnPix2

In [10]:
corono_name = 'APLC' # 'SP' or 'APLC'
CtrBtwnPix  = True
CtrBtwnPix2 = True
Pupil2dSym  = True

nlam

In [11]:
bw   = 0.1
nlam = 5

In [12]:
do_fits = True

# File reading for Pupil and Lyot stop

In [13]:
fdir = Path('./pupils/2D/').resolve()
if pupil_name == 'lvr':
    fname_pup = 'ATLAST_Aperture_nPup={0}.fits'.format(nPup,)
    fname_lys = 'ATLAST_LyotStop_nPup={0}.fits'.format(nPup,)
else:
    fname_pup = 'pupil={0}_nPup={1}.fits'.format(pupil_name, nPup,)
    fname_lys = 'pupil={0}_nPup={1}.fits'.format(pupil_name, nPup,)

fpath_pup = fdir / fname_pup
fpath_lys = fdir / fname_lys
Pupil2d    = fits.getdata(fpath_pup)
LyotStop2d = fits.getdata(fpath_lys)

In [14]:
if solver != 'gurobipy' and solver != 'stdgrb':
    solver = 'scipy'

In [15]:
params = coro.to_dict(nPup=nPup, Fmax2d = Fmax2d, nImg2d=nImg2d, nFPM = nFPM,
                 rho0=rho0, rho1=rho1, cDarkHole=cDarkHole, tau=tau, 
                 CtrBtwnPix=CtrBtwnPix, CtrBtwnPix2 = CtrBtwnPix2,
                 nlam=nlam, bw=bw,
                 Pupil2d = Pupil2d, LyotStop2d = LyotStop2d,
                 Pupil2dSym = Pupil2dSym, rMask=rMask,
                 problem_name = problem_name, 
                 solver = solver, 
                 corono_name = corono_name, pupil_name = pupil_name,
                 slvLogToConsole = slvLogToConsole,
                 slvCrossover = slvCrossover, slvMethod = slvMethod,
                 allLogToConsole = allLogToConsole,
                 MinIsland = MinIsland, FirstDerGlobalLim = FirstDerGlobalLim,
                 Binarity = Binarity, BinarityReg = BinarityReg)

# Coronagraph defintion

In [16]:
if corono_name == 'SP':
    corono0 = coro.design.SP2d(**params)
elif corono_name == 'APLC':
    corono0 = coro.design.APLC2d(**params)
else:
    raise NameError('{0}: Not an existing coronagraph!'.format(corono_name))

# Problem definition

In [17]:
if problem_name == 'MaxTau':
    # Maximization of the integrated amplitude transmission of the apodizer
    problem1 = coro.optim_2d.MaxTau(corono=corono0, **params)
elif problem_name == 'MaxContrastL1':
    # Maximization of the contrast under L1-norm
    problem1 = coro.optim_2d.MaxContrast(corono=corono0, Lnorm='L1',**params)
elif problem_name == 'MaxContrastLinf':
    # Maximization of the contrast under L-infinite norm
    problem1 = coro.optim_2d.MaxContrast(corono=corono0, Lnorm='Linf',**params)
else:
    raise NameError('{0}: Not an existing optimization problem!'.format(problem_name))

# Apodizer solutions

In [18]:
t0 = time.time()
Apod1 = problem1.solve_model()
t1 = time.time()
print('optimization time             : {0:.2f}s'.format(t1-t0))

Academic license - for non-commercial use only
Changed value of parameter Method to 2
   Prev: -1  Min: -1  Max: 5  Default: -1
optimization time             : 6.48s


# Generation of full apodizer for quarter pupil optimization

In [19]:
Apod1_2d = np.reshape(Apod1, (corono0.nPup, corono0.nPup))

if Pupil2dSym == True:
        Apod1_2dtmp =  Apod1_2d[corono0.nPup//2:, corono0.nPup//2:]
        Apod1_2d[:corono0.nPup//2, corono0.nPup//2:] = np.flip(Apod1_2dtmp, axis=0)
        Apod1_2d[:, :corono0.nPup//2]          = np.flip(Apod1_2d[:, corono0.nPup//2:], axis=1)

# Save apodizer

In [20]:
fdir = Path('./results/2D/dat_pyth').resolve() / pupil_name
if not os.path.exists(fdir):
    os.makedirs(fdir)
    
fname = problem1.get_filename() + '.fits'
fpath = fdir / fname

if do_fits is True:
    fits.writeto(fpath, Apod1_2d, overwrite=True)